In [1]:
from __future__ import annotations

import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

import numpy as np

PROJECT = Path("/home/baiyu/LearnStageConstraints")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
assert "envs/segment" in str(Path(sys.executable)), sys.executable

from envs.BarClean import load_BarClean
from experiments.artifacts import write_json
from runners.run_benchmark import run_benchmark

live_run_dir = PROJECT / "outputs/map_balanced_pooled/BarClean/method_seed_000"
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
diagnostic_root = PROJECT / "outputs/diagnostics" / f"barclean_free_tail_cuts_{stamp}"
baseline_dir = diagnostic_root / "free_tail_cuts_current"
diagnostic_root.mkdir(parents=True, exist_ok=False)
shutil.copytree(live_run_dir, baseline_dir)
print("Python:", sys.executable)
print("Diagnostic root:", diagnostic_root)


Python: /home/baiyu/miniforge3/envs/segment/bin/python
Diagnostic root: /home/baiyu/LearnStageConstraints/outputs/diagnostics/barclean_free_tail_cuts_20260828_143115


In [2]:
true_fixed_dir = diagnostic_root / "all_true_cuts"
true_fixed_dir.mkdir(parents=True, exist_ok=False)
try:
    run_benchmark(
        methods=["map_balanced_pooled"],
        datasets=["BarClean"],
        method_seeds=[0],
        config_root=PROJECT / "configs",
        outdir=true_fixed_dir / "benchmark",
        method_overrides={
            "map_balanced_pooled": {
                "fixed_true_cutpoint_indices": [0, 1, 2, 3],
                "map_demo_num_workers": 1,
                "disable_plots": True,
            }
        },
        refresh_demo_cache=False,
        resume=False,
    )
    shutil.move(str(live_run_dir), str(true_fixed_dir / "artifacts"))
finally:
    if live_run_dir.exists():
        partial_dir = diagnostic_root / "partial_live_run"
        suffix = 1
        while partial_dir.exists():
            partial_dir = diagnostic_root / f"partial_live_run_{suffix}"
            suffix += 1
        shutil.move(str(live_run_dir), str(partial_dir))
        print("Preserved partial output:", partial_dir)
    live_run_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(baseline_dir, live_run_dir)
    print("Restored current free-cut output:", live_run_dir)


[MAP] iter 000 | total=-612.046 | constraint=-612.046 | progress=0.000 | MeanAbsCutpointError=0.000 | MeanParameterError=0.027 | MeanParameterErrorRaw=0.029 | MeanStageSubgoalError=0.119 | PredictedConstraintCount=12.000 | SemanticConstraintF1=0.957 | SemanticConstraintMatchCount=11.000 | SemanticConstraintPrecision=0.917 | SemanticConstraintRecall=1.000 | TrueConstraintCount=11.000 | stage_ends=[[37, 60, 89, 102, 126], [42, 69, 111, 124, 152], [42, 67, 102, 113, 142]] | active=12 | progress_kappa=[1.4631, 6.0937, 0.468, 8.2015, 2.841]
[MAP] iter 001 | total=-411.087 | constraint=-143.331 | progress=-267.756 | MeanAbsCutpointError=0.000 | MeanParameterError=0.027 | MeanParameterErrorRaw=0.029 | MeanStageSubgoalError=0.119 | PredictedConstraintCount=12.000 | SemanticConstraintF1=0.957 | SemanticConstraintMatchCount=11.000 | SemanticConstraintPrecision=0.917 | SemanticConstraintRecall=1.000 | TrueConstraintCount=11.000 | stage_ends=[[37, 60, 89, 102, 126], [42, 69, 111, 124, 152], [42, 6

In [3]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def mode_record(artifact_dir: Path, stage: int, feature: str):
    learned = read_json(artifact_dir / "learned_constraints.json")
    return next(
        item for item in learned["feature_stage_modes"]
        if int(item["stage"]) == stage and item["feature_name"] == feature
    )

free_seg = read_json(baseline_dir / "segmentation.json")
fixed_artifacts = true_fixed_dir / "artifacts"
fixed_seg = read_json(fixed_artifacts / "segmentation.json")
free_mode = mode_record(baseline_dir, 3, "table_dist")
fixed_mode = mode_record(fixed_artifacts, 3, "table_dist")

bundle = load_BarClean(
    n_demos=9,
    source_demo_ids=[0, 1, 2],
    processed_demo_path="robot/stage_cons_iiwa14/data/processed/demo_fixed_scene3/training_5hz.npz",
    seed=2026,
)
raw_names = [str(spec["name"]) for spec in bundle.feature_schema]
table_col = raw_names.index("table_dist")
true_cuts = np.asarray(free_seg["true_cutpoints"], dtype=int)
pred_cuts = np.asarray(free_seg["predicted_cutpoints"], dtype=int)

rows = []
parts_by_region = {"absorbed_from_stage3": [], "true_stage4": [], "absorbed_from_stage5": [], "predicted_stage4": []}
for demo_id, features in enumerate(bundle.features):
    table = np.asarray(features[:, table_col], dtype=float)
    t3, t4 = int(true_cuts[demo_id, 2]), int(true_cuts[demo_id, 3])
    p3, p4 = int(pred_cuts[demo_id, 2]), int(pred_cuts[demo_id, 3])
    regions = {
        "absorbed_from_stage3": table[p3 + 1:t3 + 1],
        "true_stage4": table[t3 + 1:t4 + 1],
        "absorbed_from_stage5": table[t4 + 1:p4 + 1],
        "predicted_stage4": table[p3 + 1:p4 + 1],
    }
    for name, values in regions.items():
        if values.size:
            parts_by_region[name].append(values)
    rows.append({
        "demo": demo_id,
        "true_cp3": t3,
        "pred_cp3": p3,
        "cp3_delta": p3 - t3,
        "true_cp4": t4,
        "pred_cp4": p4,
        "cp4_delta": p4 - t4,
        "absorbed_stage3_samples": max(t3 - p3, 0),
        "absorbed_stage5_samples": max(p4 - t4, 0),
        "true_stage4_samples": t4 - t3,
        "pred_stage4_samples": p4 - p3,
        "true_stage4_table_mean": float(np.mean(regions["true_stage4"])),
        "true_stage4_table_std": float(np.std(regions["true_stage4"])),
        "pred_stage4_table_mean": float(np.mean(regions["predicted_stage4"])),
        "pred_stage4_table_std": float(np.std(regions["predicted_stage4"])),
        "pred_stage4_table_min": float(np.min(regions["predicted_stage4"])),
        "pred_stage4_table_max": float(np.max(regions["predicted_stage4"])),
    })

region_summary = {}
for name, arrays in parts_by_region.items():
    values = np.concatenate(arrays)
    region_summary[name] = {
        "n": int(values.size),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "min": float(np.min(values)),
        "q05": float(np.quantile(values, 0.05)),
        "median": float(np.median(values)),
        "q95": float(np.quantile(values, 0.95)),
        "max": float(np.max(values)),
    }

report = {
    "free_cut_config": {"fixed_true_cutpoint_indices": [0, 1]},
    "free_stage4_table_mode": free_mode,
    "all_true_cut_stage4_table_mode": fixed_mode,
    "per_demo_cut_and_table_stats": rows,
    "pooled_region_summary": region_summary,
}
write_json(diagnostic_root / "diagnosis.json", report)

print("Stage 4 table_dist")
print("free cuts:", free_mode["mode"], free_mode["mode_scores"])
print("all true cuts:", fixed_mode["mode"], fixed_mode["mode_scores"])
print()
print("demo | cp3 Δ | cp4 Δ | absorbed S3 | absorbed S5 | true S4 n | predicted S4 n | true table std | predicted table std")
for row in rows:
    print(
        row["demo"], row["cp3_delta"], row["cp4_delta"],
        row["absorbed_stage3_samples"], row["absorbed_stage5_samples"],
        row["true_stage4_samples"], row["pred_stage4_samples"],
        round(row["true_stage4_table_std"], 5),
        round(row["pred_stage4_table_std"], 5),
        sep=" | ",
    )
print()
print("Pooled table_dist regions:")
for name, stats in region_summary.items():
    print(name, {key: round(value, 6) if isinstance(value, float) else value for key, value in stats.items()})
print("Saved:", diagnostic_root / "diagnosis.json")


Stage 4 table_dist
free cuts: lower_bound {'inactive': 0.30318599968037735, 'target_value': 0.00047831443594781515, 'lower_bound': 0.6912684615705994, 'upper_bound': 0.005067224313075501}
all true cuts: target_value {'inactive': 0.0006610250048138969, 'target_value': 0.997148269360652, 'lower_bound': 0.0010940124686006076, 'upper_bound': 0.0010966931659335487}

demo | cp3 Δ | cp4 Δ | absorbed S3 | absorbed S5 | true S4 n | predicted S4 n | true table std | predicted table std
0 | -7 | 7 | 7 | 7 | 13 | 27 | 0.00049 | 0.01726
1 | -13 | 6 | 13 | 6 | 13 | 32 | 0.00057 | 0.01562
2 | -9 | 3 | 9 | 3 | 11 | 23 | 0.00038 | 0.00781

Pooled table_dist regions:
absorbed_from_stage3 {'n': 29, 'mean': 0.07382, 'std': 0.014345, 'min': 0.063794, 'q05': 0.064048, 'median': 0.067767, 'q95': 0.101395, 'max': 0.12545}
true_stage4 {'n': 37, 'mean': 0.064047, 'std': 0.000683, 'min': 0.062515, 'q05': 0.063054, 'median': 0.06402, 'q95': 0.065218, 'max': 0.065433}
absorbed_from_stage5 {'n': 16, 'mean': 0.08441